# Lecture: Clinical Research Informatics - Sommer Semester 2026
Exercise Sheet: 5

Additional information:   
used additional packages: pandas 2.3.3   
dataset used: Coding Data 02 - standard.zip

In [2]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")

# Functions

In [3]:
def calculate_age_in_years(birth_dates: pd.Series, reference_dates: pd.Series) -> pd.Series:
    """Calculate age in whole years between birth dates and reference dates."""
    return ((reference_dates - birth_dates).dt.days // 365)


def add_current_age(patients: pd.DataFrame, reference_date: pd.Timestamp | None = None) -> pd.DataFrame:
    """Add current age to the patient DataFrame."""
    patients = patients.copy()
    reference_date = reference_date or pd.Timestamp(date.today())
    end_dates = patients["DEATHDATE"].fillna(reference_date)
    patients["current_age"] = calculate_age_in_years(patients["BIRTHDATE"], end_dates)
    return patients


def add_age_at_condition_start(conditions: pd.DataFrame, patients: pd.DataFrame) -> pd.DataFrame:
    """Add patient age at condition start to the conditions DataFrame."""
    conditions = conditions.copy()
    birthdates = patients.set_index("Id")["BIRTHDATE"]
    conditions["age_at_start"] = calculate_age_in_years(
        conditions["PATIENT"].map(birthdates),
        conditions["START"],
    )
    return conditions


def filter_patients_by_age(
    patients: pd.DataFrame,
    min_age: int,
    max_age: int,
    age_column: str = "current_age",
) -> pd.DataFrame:
    """Filter patient records to those with age within the given range (inclusive)."""
    return patients[(patients[age_column] >= min_age) & (patients[age_column] <= max_age)].copy()

# Section 1: Data Exploration and Preparation

### 1.1 Import datasets and show dimensions

In [4]:
DATA_FILES = {
    "patients": "patients.csv",
    "conditions": "conditions_reduced.csv",
    "careplans": "careplans.csv",
    "allergies": "allergies.csv",
    "medications": "medications_reduced.csv",
    "encounters": "encounter_reduced.csv",
    "observations": "observations_reduced.csv",
}

date_columns = {
    "patients": ["BIRTHDATE", "DEATHDATE"],
    "conditions": ["START", "STOP"],
    "careplans": ["START", "STOP"],
    "allergies": ["START", "STOP"],
    "medications": ["START", "STOP"],
    "encounters": ["START", "STOP"],
    "observations": ["DATE"],
}

dataframes = {}
for name, filename in DATA_FILES.items():
    df = pd.read_csv(DATA_DIR / filename, parse_dates=date_columns[name])
    dataframes[name] = df
    print("{}: {} variables (columns), {} entries (rows)".format(filename, df.shape[1], df.shape[0]))

patients = dataframes["patients"]
conditions = dataframes["conditions"]

patients.csv: 27 variables (columns), 58307 entries (rows)
conditions_reduced.csv: 6 variables (columns), 481427 entries (rows)
careplans.csv: 9 variables (columns), 204096 entries (rows)
allergies.csv: 15 variables (columns), 51846 entries (rows)
medications_reduced.csv: 13 variables (columns), 131382 entries (rows)
encounter_reduced.csv: 15 variables (columns), 78274 entries (rows)
observations_reduced.csv: 9 variables (columns), 1048575 entries (rows)


### 1.2 Add age columns and print averages

In [5]:
patients = add_current_age(patients)
conditions = add_age_at_condition_start(conditions, patients)

print("Average patient age: {:.2f} years".format(patients["current_age"].mean()))
print("Average age at condition start: {:.2f} years".format(conditions["age_at_start"].mean()))

Average patient age: 43.55 years
Average age at condition start: 46.02 years


### 1.3 Filter patients aged 6–12 years

In [6]:
records_before = len(patients)
patients_children_6_12 = filter_patients_by_age(patients, min_age=6, max_age=12)
records_after = len(patients_children_6_12)

print("Records before filtering: {}".format(records_before))
print("Records after filtering:  {}".format(records_after))

Records before filtering: 58307
Records after filtering:  4125
